In [2]:
import os
import glob
import pandas as pd

### Download dataset

In [ ]:
%%bash

data_dir=./data
mkdir -p $data_dir
cd $data_dir

# download targets data
rsync -rpltvz \
  rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.06/output/target \
  .

# download diseases or phenotypes data
rsync -rpltvz \
  rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.06/output/disease \
  .

# download drug data
rsync -rpltvz --delete \
  rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.06/output/drug_molecule \
  .
  
rsync -rpltvz --delete \
  rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.06/output/drug_mechanism_of_action \
  .

# download target–disease association data
rsync -rpltvz \
  rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.06/output/association_by_datasource_direct \
  .

# download evidence
rsync -rpltvz \
  rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.06/output/evidence_europepmc/part-00000-621a7bae-da68-4f0d-b14a-61d21b19204c-c000.snappy.parquet \
  evidence_part0/
  
rsync -rpltvz \
  rsync.ebi.ac.uk::pub/databases/opentargets/platform/26.06/output/evidence_clinical_precedence \
  .

### Inspect and process dataset

In [3]:
data_dir="./data"
os.chdir(data_dir)
nodes_dir = "graph_data/nodes"
edges_dir = "graph_data/edges"
os.makedirs(nodes_dir, exist_ok=True)
os.makedirs(edges_dir, exist_ok=True)
glob.glob('*/*parquet')

['evidence_part0/part-00000-621a7bae-da68-4f0d-b14a-61d21b19204c-c000.snappy.parquet',
 'drug_mechanism_of_action/part-00000-10b94b1b-f29a-440c-98e0-c91862b6d2a8-c000.snappy.parquet',
 'drug_mechanism_of_action/part-00001-10b94b1b-f29a-440c-98e0-c91862b6d2a8-c000.snappy.parquet',
 'disease/disease.parquet',
 'drug_molecule/part-00000-5581d2c3-048b-4097-934f-51cec06d7ff0-c000.snappy.parquet',
 'association_by_datasource_direct/part-00013-02673c35-4621-4b9d-b289-f04a44e86cbc-c000.snappy.parquet',
 'association_by_datasource_direct/part-00011-02673c35-4621-4b9d-b289-f04a44e86cbc-c000.snappy.parquet',
 'association_by_datasource_direct/part-00003-02673c35-4621-4b9d-b289-f04a44e86cbc-c000.snappy.parquet',
 'association_by_datasource_direct/part-00005-02673c35-4621-4b9d-b289-f04a44e86cbc-c000.snappy.parquet',
 'association_by_datasource_direct/part-00004-02673c35-4621-4b9d-b289-f04a44e86cbc-c000.snappy.parquet',
 'association_by_datasource_direct/part-00007-02673c35-4621-4b9d-b289-f04a44e86c

#### Targets

In [3]:
# targets
targets_df = pd.read_parquet("target")
print(targets_df.head())
summary = {
    "rows": len(targets_df),
    "targets":targets_df["id"].nunique(),
}
print(summary)
print(targets_df.columns.tolist())

                id approvedSymbol         biotype alternativeGenes  \
0  ENSG00000083454          P2RX5  protein_coding             None   
1  ENSG00000087087           SRRT  protein_coding             None   
2  ENSG00000092201        SUPT16H  protein_coding             None   
3  ENSG00000100567          PSMA3  protein_coding             None   
4  ENSG00000105707            HPN  protein_coding             None   

                                     genomicLocation  \
0  {'chromosome': '17', 'start': 3672199, 'end': ...   
1  {'chromosome': '7', 'start': 100875065, 'end':...   
2  {'chromosome': '14', 'start': 21351472, 'end':...   
3  {'chromosome': '14', 'start': 58244740, 'end':...   
4  {'chromosome': '19', 'start': 35040506, 'end':...   

                                       transcriptIds  \
0  [ENST00000547933, ENST00000853234, ENST0000055...   
1  [ENST00000965068, ENST00000860832, ENST0000096...   
2  [ENST00000556217, ENST00000555943, ENST0000089...   
3  [ENST0000055367

In [ ]:
targets_df = targets_df[
    [
        "id",
        "approvedSymbol",
        "approvedName"
    ]
]
targets_df.columns = [
    "target_id",
    "symbol",
    "name"
]

targets_df.to_parquet(
    nodes_dir + "/targets.parquet",
    index=False
)
targets_df.head()

,target_id,symbol,name
0,ENSG00000083454,P2RX5,purinergic receptor P2X 5
1,ENSG00000087087,SRRT,"serrate, RNA effector molecule"
2,ENSG00000092201,SUPT16H,"SPT16 homolog, facilitates chromatin remodelin..."
3,ENSG00000100567,PSMA3,proteasome 20S subunit alpha 3
4,ENSG00000105707,HPN,hepsin


#### Diseases

In [5]:
# diseases or phenotypes
disease_df = pd.read_parquet("disease")
print(disease_df.head())
summary = {
    "rows": len(disease_df),
    "diseases or phenotypes": disease_df["id"].nunique(),
}
print(summary)
print(disease_df.columns.tolist())

           id                                       code  \
0  GO_0000050  http://purl.obolibrary.org/obo/GO_0000050   
1  GO_0000096  http://purl.obolibrary.org/obo/GO_0000096   
2  GO_0000226  http://purl.obolibrary.org/obo/GO_0000226   
3  GO_0000278  http://purl.obolibrary.org/obo/GO_0000278   
4  GO_0001503  http://purl.obolibrary.org/obo/GO_0001503   

                                    name  \
0                             urea cycle   
1    sulfur amino acid metabolic process   
2  microtubule cytoskeleton organization   
3                     mitotic cell cycle   
4                           ossification   

                                         description  \
0  The sequence of reactions by which arginine is...   
1  The chemical reactions and pathways involving ...   
2  A process that is carried out at the cellular ...   
3  Progression through the phases of the mitotic ...   
4  The formation of bone or of a bony substance, ...   

                                     

In [ ]:
disease = disease_df[
    [
        "id",
        "name"
    ]
]

disease.columns = [
    "disease_id",
    "name"
]

disease.to_parquet(
    nodes_dir + "/diseases.parquet",
    index=False
)
print(disease.tail())

           disease_id                                               name
47075  Orphanet_99942  Autosomal dominant Charcot-Marie-Tooth disease...
47076  Orphanet_99943  Autosomal dominant Charcot-Marie-Tooth disease...
47077  Orphanet_99945  Autosomal dominant Charcot-Marie-Tooth disease...
47078  Orphanet_99946  Autosomal dominant Charcot-Marie-Tooth disease...
47079  Orphanet_99947  Autosomal dominant Charcot-Marie-Tooth disease...


#### Drugs

In [7]:
# drug
drug_df = pd.read_parquet("drug_molecule")
print(drug_df.head())
summary = {
    "rows": len(drug_df),
    "drugs": drug_df["id"].nunique(),
}
print(summary)
print(drug_df.columns.tolist())

             id                                    canonicalSmiles  \
0    CHEMBL1007  CC(C)C[C@H](NC(=O)CNC(=O)[C@H](Cc1ccc(O)cc1)NC...   
1  CHEMBL103667  Cc1ccc(-n2nc(C(C)(C)C)cc2NC(=O)Nc2ccc(OCCN3CCO...   
2    CHEMBL1037                           N=C(N)NCC1COC2(CCCCC2)O1   
3    CHEMBL1043                    Nc1ccc(S(=O)(=O)c2ccc(N)cc2)cc1   
4  CHEMBL105318    Cc1cn([C@H]2C[C@H](F)[C@@H](CO)O2)c(=O)[nH]c1=O   

                      inchiKey  \
0  XLXSAKCOAKORKW-AQJXLSMYSA-N   
1  MVCOAUNKQVWQHZ-UHFFFAOYSA-N   
2  HPBNRIOWIXYZFK-UHFFFAOYSA-N   
3  MQJKPEGWNLWLTK-UHFFFAOYSA-N   
4  UXCAQJAQSWSNPQ-XLPZGREQSA-N   

                                            molblock        drugType  \
0  \n     RDKit          2D\n\n 85 90  0  0  1  0...         Protein   
1  \n     RDKit          2D\n\n 39 43  0  0  0  0...  Small molecule   
2  \n  SciTegic12091219242D\n\n 15 16  0  0  0  0...  Small molecule   
3  \n     RDKit          2D\n\n 17 18  0  0  0  0...  Small molecule   
4  \n     RDKi

In [ ]:
drug_df = drug_df[
    [
        "id",
        "name",
        "drugType",
        "canonicalSmiles",
        "maximumClinicalStage",
        "description"
    ]
]

drug_df.columns = [
    "drug_id",
    "name",
    "drug_type",
    "smiles",
    "clinical_stage",
    "description"
]

drug_df.to_parquet(
    nodes_dir + "/drugs.parquet",
    index=False
)
drug_df.head()

,drug_id,name,drug_type,smiles,clinical_stage,description
0,CHEMBL1007,GONADORELIN,Protein,CC(C)C[C@H](NC(=O)CNC(=O)[C@H](Cc1ccc(O)cc1)NC...,APPROVAL,Protein drug with a maximum clinical stage of ...
1,CHEMBL103667,DORAMAPIMOD,Small molecule,Cc1ccc(-n2nc(C(C)(C)C)cc2NC(=O)Nc2ccc(OCCN3CCO...,PHASE_2,Small molecule drug with a maximum clinical st...
2,CHEMBL1037,GUANADREL,Small molecule,N=C(N)NCC1COC2(CCCCC2)O1,UNKNOWN,Small molecule drug.
3,CHEMBL1043,DAPSONE,Small molecule,Nc1ccc(S(=O)(=O)c2ccc(N)cc2)cc1,APPROVAL,Small molecule drug with a maximum clinical st...
4,CHEMBL105318,ALOVUDINE,Small molecule,Cc1cn([C@H]2C[C@H](F)[C@@H](CO)O2)c(=O)[nH]c1=O,PHASE_2,Small molecule drug with a maximum clinical st...


In [9]:
# drug-target
drug_mechanism_df = pd.read_parquet("drug_mechanism_of_action")
print(drug_mechanism_df.head())
summary = {
    "rows": len(drug_mechanism_df),
    "actionType": drug_mechanism_df["actionType"].nunique(),
}
print(summary)
print(drug_mechanism_df.columns.tolist())

  actionType               mechanismOfAction                chemblIds  \
0  ACTIVATOR  Acetylcholinesterase activator  [CHEMBL748, CHEMBL1420]   
1  ACTIVATOR      Antithrombin-III activator          [CHEMBL1201448]   
2  ACTIVATOR      Antithrombin-III activator          [CHEMBL1201513]   
3  ACTIVATOR      Antithrombin-III activator          [CHEMBL1201534]   
4  ACTIVATOR      Antithrombin-III activator          [CHEMBL1201657]   

             targetName      targetType            targets  \
0  Acetylcholinesterase  single protein  [ENSG00000087085]   
1      Antithrombin-III  single protein  [ENSG00000117601]   
2      Antithrombin-III  single protein  [ENSG00000117601]   
3      Antithrombin-III  single protein  [ENSG00000117601]   
4      Antithrombin-III  single protein  [ENSG00000117601]   

                                          references  
0  [{'source': 'DailyMed', 'ids': ['setid=a16ac22...  
1  [{'source': 'DailyMed', 'ids': ['setid=cbc26e3...  
2  [{'source': 'DailyMe

In [ ]:
drug_mechanism_df = drug_mechanism_df [
    [
        "chemblIds",
        "targets",
        "actionType",
        "mechanismOfAction"
    ]
]

drug_mechanism_df = drug_mechanism_df.explode("chemblIds")
drug_mechanism_df = drug_mechanism_df.explode("targets")


drug_mechanism_df.columns = [
    "drug_id",
    "target_id",
    "action_type",
    "mechanism"
]


drug_mechanism_df.to_parquet(
    edges_dir + "/drug_target.cparquet",
    index=False
)

print(drug_mechanism_df.head())

         drug_id        target_id action_type                       mechanism
0      CHEMBL748  ENSG00000087085   ACTIVATOR  Acetylcholinesterase activator
0     CHEMBL1420  ENSG00000087085   ACTIVATOR  Acetylcholinesterase activator
1  CHEMBL1201448  ENSG00000117601   ACTIVATOR      Antithrombin-III activator
2  CHEMBL1201513  ENSG00000117601   ACTIVATOR      Antithrombin-III activator
3  CHEMBL1201534  ENSG00000117601   ACTIVATOR      Antithrombin-III activator


#### Target-Disease

In [11]:
# target-disease
target_disease_df = pd.read_parquet(
    "association_by_datasource_direct",
        columns=[
        "diseaseId",
        "targetId",
        "aggregationValue",
        "associationScore",
        "evidenceCount"
    ])

print(target_disease_df.head())
summary = {
    "rows": len(target_disease_df),
    "targets": target_disease_df["targetId"].nunique(),
    "diseases": target_disease_df["diseaseId"].nunique(),
    "target_disease_pairs": (
        target_disease_df[
            ["targetId", "diseaseId"]
        ]
        .drop_duplicates()
        .shape[0]
    ),
    "datasources": target_disease_df["aggregationValue"].nunique(),
}
print(summary)

print(target_disease_df .columns.tolist())

     diseaseId         targetId aggregationValue  associationScore  \
0  EFO_0000180  ENSG00000020633        europepmc          0.030415   
1  EFO_0000180  ENSG00000081985        europepmc          0.012166   
2  EFO_0000180  ENSG00000086065        europepmc          0.076037   
3  EFO_0000180  ENSG00000091513        europepmc          0.079417   
4  EFO_0000180  ENSG00000102265        europepmc          0.115069   

   evidenceCount  
0            1.0  
1            1.0  
2            2.0  
3           16.0  
4            3.0  
{'rows': 7842921, 'targets': 31459, 'diseases': 26517, 'target_disease_pairs': 7554123, 'datasources': 20}
['diseaseId', 'targetId', 'aggregationValue', 'associationScore', 'evidenceCount']


In [ ]:
target_disease_df = target_disease_df[
    [
        "diseaseId",
        "targetId",
        "aggregationValue",
        "associationScore",
        "evidenceCount"
    ]
]

target_disease_df.columns = [
    "disease_id",
    "target_id",
    "datasource",
    "score",
    "evidence_count"
]

target_disease_df.to_parquet(
    edges_dir + "/disease_target.parquet",
    index=False
)
print(target_disease_df.head())

    disease_id        target_id datasource     score  evidence_count
0  EFO_0000180  ENSG00000020633  europepmc  0.030415             1.0
1  EFO_0000180  ENSG00000081985  europepmc  0.012166             1.0
2  EFO_0000180  ENSG00000086065  europepmc  0.076037             2.0
3  EFO_0000180  ENSG00000091513  europepmc  0.079417            16.0
4  EFO_0000180  ENSG00000102265  europepmc  0.115069             3.0


#### Clinical evidence

In [8]:
# target-disease
clinical_evidence_df = pd.read_parquet("evidence_clinical_precedence")

print(clinical_evidence_df.head())
print(clinical_evidence_df.columns.tolist())

          targetId                                        id  \
0  ENSG00000101162  003c319d6577e8d841c1bcfee224e122d90d995d   
1  ENSG00000132170  0045d9a6cc0f0f922df8942e12a73be1519bd27f   
2  ENSG00000198900  004c8b6769e89e0eef3828abc084d1b006b5ea2a   
3  ENSG00000022355  00899b681d87df6050558e64dd34cb33b4bc8691   
4  ENSG00000123416  01735c29b08a148c0e4d62a958f8a0d1421eb79b   

  targetFromSourceId diseaseFromSourceMappedId  \
0    ENSG00000101162             MONDO_0007254   
1    ENSG00000132170             MONDO_0005101   
2    ENSG00000198900             MONDO_0015686   
3    ENSG00000022355             MONDO_0005383   
4    ENSG00000123416             MONDO_0007254   

                       clinicalReportId clinicalStage trialWhyStopped  \
0                           nct00785291       PHASE_3            None   
1  75a96561-6f52-4c11-a105-391fbbfe4827      APPROVAL            None   
2                           nct00003523       PHASE_2            None   
3  31dacbaf-4245-4172-

In [ ]:
clinical_evidence_df = clinical_evidence_df[
    [
        "id",
        "targetId",
        "diseaseId",
        "drugId",
        "clinicalReportId",
        "clinicalStage",
        "score",
        "literature"
    ]
]

clinical_evidence_df.columns = [
        "evidence_id",
        "target_id",
        "disease_id",
        "drug_id",
        "clinical_report_id",
        "clinical_stage",
        "score",
        "literature"
    ]

clinical_evidence_df.to_parquet(
    edges_dir + "/clinical_evidence.parquet",
    index=False
)

print(clinical_evidence_df.head())

                                evidence_id        target_id     disease_id  \
0  003c319d6577e8d841c1bcfee224e122d90d995d  ENSG00000101162  MONDO_0007254   
1  0045d9a6cc0f0f922df8942e12a73be1519bd27f  ENSG00000132170  MONDO_0005101   
2  004c8b6769e89e0eef3828abc084d1b006b5ea2a  ENSG00000198900  MONDO_0015686   
3  00899b681d87df6050558e64dd34cb33b4bc8691  ENSG00000022355  MONDO_0005383   
4  01735c29b08a148c0e4d62a958f8a0d1421eb79b  ENSG00000123416  MONDO_0007254   

         drug_id                    clinical_report_id clinical_stage  score  \
0  CHEMBL1201752                           nct00785291        PHASE_3   0.70   
1  CHEMBL1201013  75a96561-6f52-4c11-a105-391fbbfe4827       APPROVAL   1.00   
2   CHEMBL274070                           nct00003523        PHASE_2   0.20   
3      CHEMBL661  31dacbaf-4245-4172-9da4-514be309f341       APPROVAL   1.00   
4  CHEMBL3545252                           nct05287308        UNKNOWN   0.01   

                                 literature 

: 